In [ ]:
import anndata as ad

INPUT = "/home/csb/sampoorn/emt_scc/12_scc/emt_scc_ideas_input.h5ad"

adata = ad.read_h5ad(INPUT)

print(adata)
print("Shape:", adata.shape)

AnnData object with n_obs × n_vars = 23959 × 20358
Shape: (23959, 20358)


In [ ]:
import pandas as pd

annotation_df = pd.read_csv(
    "EMT_celltype_annotations.csv",
    index_col="cell_id"
)

# Make sure both use strings for cell IDs
annotation_df.index = annotation_df.index.astype(str)
adata.obs_names = adata.obs_names.astype(str)

# Merge annotation by cell ID
adata.obs["celltype_annotation"] = (
    annotation_df["celltype_annotation"]
    .reindex(adata.obs_names)
)

print(adata.obs["celltype_annotation"].value_counts(dropna=False))

celltype_annotation
Early-Hybrid    9985
Late-EMT        6220
Epithelial      4146
Hybrid          3608
Name: count, dtype: int64


In [ ]:
print("Total cells:", adata.n_obs)
print(
    "Annotated cells:",
    adata.obs["celltype_annotation"].notna().sum()
)
print(
    "Missing annotations:",
    adata.obs["celltype_annotation"].isna().sum()
)

print("\nFirst 10 cells:")
print(adata.obs[["celltype_annotation"]].head(10))

Total cells: 23959
Annotated cells: 23959
Missing annotations: 0

First 10 cells:
                   celltype_annotation
AAACCCACACGGCACT-1              Hybrid
AAACGAAAGCTAATGA-1            Late-EMT
AAACGAAAGGCTAGCA-1            Late-EMT
AAACGAACAATAGTCC-1          Epithelial
AAACGAACACTAGAGG-1            Late-EMT
AAACGAATCCGTTGGG-1              Hybrid
AAACGAATCGCCGAAC-1              Hybrid
AAACGCTAGGCATCAG-1            Late-EMT
AAACGCTCAAAGACGC-1            Late-EMT
AAAGGATCACACTTAG-1              Hybrid




import sys
import matplotlib.pyplot as plt

# IDEAS source
sys.path.insert(0, "/home/csb/sampoorn/IDEAS")

from computeID.functions_IDEAS import compute_ID


# ---------------------------------------------------------
# Balanced global 2NN ID
# ---------------------------------------------------------
# Smallest population = Hybrid = 3608
# 80% of 3608 = 2886
# Therefore every state uses 2886 cells per replicate

id_reps = compute_ID(
    adata,
    group="celltype_annotation",
    method="2nn",
    sample_size=2886,
    n_samples=30,
    normalization=True,
    norm_sum=1e6,
    full_output=True,
    id_score=False
)

print(id_reps)
print("Shape:", id_reps.shape)


# ---------------------------------------------------------
# Box plot
# ---------------------------------------------------------

order = [
    "Epithelial",
    "Early-Hybrid",
    "Hybrid",
    "Late-EMT"
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    [
        id_reps[state].dropna().values
        for state in order
    ],
    labels=order,
    showfliers=False
)

# Overlay the 30 individual balanced estimates
for i, state in enumerate(order, start=1):

    values = id_reps[state].dropna().values

    x = (
        i
        + 0.04 * (
            __import__("numpy").random.default_rng(42)
            .uniform(-1, 1, len(values))
        )
    )

    plt.scatter(
        x,
        values,
        s=20,
        alpha=0.6
    )

plt.ylabel("Global Intrinsic Dimension")
plt.xlabel("EMT State")

plt.xticks(
    range(1, 5),
    order,
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()



In [ ]:
import sys
import matplotlib.pyplot as plt

# IDEAS source
sys.path.insert(0, "/home/csb/sampoorn/IDEAS")

from computeID.functions_IDEAS import compute_ID


# ---------------------------------------------------------
# Balanced global 2NN ID
# ---------------------------------------------------------
# Smallest population = Hybrid = 3608
# 80% of 3608 = 2886
# Therefore every state uses 2886 cells per replicate

id_reps = compute_ID(
    adata,
    group="celltype_annotation",
    method="2nn",
    sample_size=2886,
    n_samples=100,
    normalization=True,
    norm_sum=1e6,
    full_output=True,
    id_score=False
)

print(id_reps)
print("Shape:", id_reps.shape)


# ---------------------------------------------------------
# Box plot
# ---------------------------------------------------------

order = [
    "Epithelial",
    "Early-Hybrid",
    "Hybrid",
    "Late-EMT"
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    [
        id_reps[state].dropna().values
        for state in order
    ],
    labels=order,
    showfliers=False
)

# Overlay the 30 individual balanced estimates
for i, state in enumerate(order, start=1):

    values = id_reps[state].dropna().values

    x = (
        i
        + 0.04 * (
            __import__("numpy").random.default_rng(42)
            .uniform(-1, 1, len(values))
        )
    )

    plt.scatter(
        x,
        values,
        s=20,
        alpha=0.6
    )

plt.ylabel("Global Intrinsic Dimension")
plt.xlabel("EMT State")

plt.xticks(
    range(1, 5),
    order,
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()

Computing Intrinsic Dimension:   0%|          | 0/4 [00:00<?, ?it/s]

Computing ID for celltype_annotation Early-Hybrid:


Computing Intrinsic Dimension:  25%|██▌       | 1/4 [05:33<16:39, 333.24s/it]

Computing ID for celltype_annotation Epithelial:


Computing Intrinsic Dimension:  50%|█████     | 2/4 [09:18<08:59, 269.87s/it]

Computing ID for celltype_annotation Hybrid:


Computing Intrinsic Dimension:  75%|███████▌  | 3/4 [12:13<03:46, 226.58s/it]

Computing ID for celltype_annotation Late-EMT:


Computing Intrinsic Dimension: 100%|██████████| 4/4 [15:12<00:00, 228.11s/it]


     Early-Hybrid  Epithelial     Hybrid   Late-EMT
1       23.194570   22.990030  20.633768  20.577616
2       23.737898   23.455517  21.581519  19.843677
3       22.927989   23.271810  20.524388  20.497941
4       22.989662   23.182688  20.955964  19.910717
5       22.641082   22.728951  21.258430  20.764623
..            ...         ...        ...        ...
96      23.656381   23.157497  21.327437  20.080240
97      23.528646   23.788425  20.631407  20.114618
98      23.429301   23.302415  20.778968  20.515302
99      22.594223   23.015379  20.920344  20.224729
100     23.049114   22.645874  21.001033  20.393111

[100 rows x 4 columns]
Shape: (100, 4)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

order = [
    "Epithelial",
    "Early-Hybrid",
    "Hybrid",
    "Late-EMT"
]

plt.figure(figsize=(8, 5))

plt.boxplot(
    [
        id_reps[state].dropna().values
        for state in order
    ],
    labels=order,
    showfliers=False
)

# Overlay individual replicate scores
rng = np.random.default_rng(42)

for i, state in enumerate(order, start=1):
    values = id_reps[state].dropna().values

    x = i + rng.uniform(
        -0.04,
        0.04,
        len(values)
    )

    plt.scatter(
        x,
        values,
        s=20,
        alpha=0.5
    )

plt.ylabel("ID Score")
plt.xlabel("EMT State")

plt.xticks(
    range(1, 5),
    order,
    rotation=30,
    ha="right"
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import kruskal, rankdata
from statsmodels.stats.multitest import multipletests
from itertools import combinations
from scipy.stats import norm


# ============================================================
# SETTINGS
# ============================================================

order = [
    "Epithelial",
    "Early-Hybrid",
    "Hybrid",
    "Late-EMT"
]

# Extract the 100 replicate ID values
data = {
    state: id_reps[state].dropna().values
    for state in order
}


# ============================================================
# 1. OMNIBUS KRUSKAL-WALLIS TEST
# ============================================================

H, p_kw = kruskal(
    *[data[state] for state in order]
)

print("==============================================")
print("KRUSKAL-WALLIS TEST")
print("==============================================")
print(f"H statistic = {H:.4f}")
print(f"p-value     = {p_kw:.4e}")


# ============================================================
# 2. DUNN'S POST-HOC TEST
# ============================================================
#
# Dunn's test using pooled ranks.
# P-values are corrected using Benjamini-Hochberg FDR.
# ============================================================

all_values = np.concatenate(
    [data[state] for state in order]
)

groups = np.concatenate(
    [
        np.repeat(state, len(data[state]))
        for state in order
    ]
)

# Rank all observations together
ranks = rankdata(all_values)

rank_df = pd.DataFrame({
    "group": groups,
    "rank": ranks
})

N = len(all_values)

# Mean rank for each state
mean_ranks = (
    rank_df
    .groupby("group")["rank"]
    .mean()
)

# Number of observations per group
n_groups = {
    state: len(data[state])
    for state in order
}


# ------------------------------------------------------------
# Tie correction
# ------------------------------------------------------------

_, tie_counts = np.unique(
    all_values,
    return_counts=True
)

tie_term = np.sum(
    tie_counts**3 - tie_counts
)

tie_correction = (
    1 -
    tie_term / (N**3 - N)
)


# ------------------------------------------------------------
# Pairwise Dunn tests
# ------------------------------------------------------------

results = []

for g1, g2 in combinations(order, 2):

    n1 = n_groups[g1]
    n2 = n_groups[g2]

    rank_difference = (
        mean_ranks[g1] -
        mean_ranks[g2]
    )

    SE = np.sqrt(
        (
            N * (N + 1) / 12
        ) *
        (
            1 / n1 +
            1 / n2
        ) *
        tie_correction
    )

    z = rank_difference / SE

    p = 2 * norm.sf(abs(z))

    results.append({
        "group1": g1,
        "group2": g2,
        "z": z,
        "p_uncorrected": p
    })


dunn_df = pd.DataFrame(results)


# ============================================================
# 3. BENJAMINI-HOCHBERG FDR
# ============================================================

dunn_df["p_FDR"] = multipletests(
    dunn_df["p_uncorrected"],
    method="fdr_bh"
)[1]


# ============================================================
# 4. SIGNIFICANCE LABELS
# ============================================================

def significance_label(p):

    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


dunn_df["significance"] = (
    dunn_df["p_FDR"]
    .apply(significance_label)
)


# ============================================================
# 5. PRINT COMPLETE STATISTICS
# ============================================================

print("\n==============================================")
print("DUNN POST-HOC TEST — BH FDR CORRECTED")
print("==============================================")

print(
    dunn_df.to_string(index=False)
)


# ============================================================
# 6. CREATE P-VALUE MATRIX
# ============================================================

p_matrix = pd.DataFrame(
    np.nan,
    index=order,
    columns=order
)

sig_matrix = pd.DataFrame(
    "",
    index=order,
    columns=order
)

for _, row in dunn_df.iterrows():

    g1 = row["group1"]
    g2 = row["group2"]

    p_matrix.loc[g2, g1] = row["p_FDR"]

    sig_matrix.loc[g2, g1] = (
        significance_label(row["p_FDR"])
    )


# ============================================================
# 7. PLOT — DUNN FDR HEATMAP
# ============================================================

plt.figure(figsize=(9, 7))

# Mask upper triangle
mask = np.triu(
    np.ones_like(
        p_matrix.astype(float),
        dtype=bool
    )
)

sns.heatmap(
    p_matrix.astype(float),
    mask=mask,
    annot=False,
    cmap="viridis_r",
    vmin=0,
    vmax=0.05,
    linewidths=2,
    linecolor="white",
    cbar_kws={
        "label": "Dunn test BH-adjusted p-value"
    }
)

# Add formatted p-values manually
for i in range(len(order)):
    for j in range(len(order)):

        if i > j:

            p = p_matrix.iloc[i, j]
            sig = sig_matrix.iloc[i, j]

            if p < 0.001:
                text = f"{p:.2e}\n{sig}"
            else:
                text = f"{p:.3f}\n{sig}"

            plt.text(
                j + 0.5,
                i + 0.5,
                text,
                ha="center",
                va="center",
                fontsize=10
            )


plt.title(
    "IDEAS Score: Pairwise Dunn Comparisons",
    fontsize=14
)

plt.xlabel("")
plt.ylabel("")

plt.xticks(
    np.arange(len(order)) + 0.5,
    order,
    rotation=30,
    ha="right"
)

plt.yticks(
    np.arange(len(order)) + 0.5,
    order,ss
    rotation=0
)

plt.tight_layout()
plt.show()

KRUSKAL-WALLIS TEST
H statistic = 313.6114
p-value     = 1.1263e-67

DUNN POST-HOC TEST — BH FDR CORRECTED
      group1       group2         z  p_uncorrected        p_FDR significance
  Epithelial Early-Hybrid -0.270331   7.869057e-01 7.869057e-01           ns
  Epithelial       Hybrid 10.207133   1.842287e-24 2.763430e-24         ****
  Epithelial     Late-EMT 13.986873   1.874766e-44 5.624297e-44         ****
Early-Hybrid       Hybrid 10.477464   1.096450e-25 2.192900e-25         ****
Early-Hybrid     Late-EMT 14.257203   4.043355e-46 2.426013e-45         ****
      Hybrid     Late-EMT  3.779740   1.569923e-04 1.883907e-04          ***


In [ ]:
def rank_biserial(x, y):

    x = np.asarray(x)
    y = np.asarray(y)

    combined = np.concatenate([x, y])

    ranks = rankdata(combined)

    R1 = np.sum(
        ranks[:len(x)]
    )

    U1 = (
        R1 -
        len(x) * (len(x) + 1) / 2
    )

    return (
        2 * U1 /
        (len(x) * len(y))
        - 1
    )


effect_results = []

for g1, g2 in combinations(order, 2):

    effect = rank_biserial(
        data[g1],
        data[g2]
    )

    effect_results.append({
        "group1": g1,
        "group2": g2,
        "rank_biserial": effect
    })

effect_df = pd.DataFrame(effect_results)

dunn_df = dunn_df.merge(
    effect_df,
    on=["group1", "group2"]
)

print(
    dunn_df[
        [
            "group1",
            "group2",
            "z",
            "p_uncorrected",
            "p_FDR",
            "rank_biserial",
            "significance"
        ]
    ].to_string(index=False)
)

      group1       group2         z  p_uncorrected        p_FDR  rank_biserial significance
  Epithelial Early-Hybrid -0.270331   7.869057e-01 7.869057e-01        -0.0442           ns
  Epithelial       Hybrid 10.207133   1.842287e-24 2.763430e-24         1.0000         ****
  Epithelial     Late-EMT 13.986873   1.874766e-44 5.624297e-44         1.0000         ****
Early-Hybrid       Hybrid 10.477464   1.096450e-25 2.192900e-25         1.0000         ****
Early-Hybrid     Late-EMT 14.257203   4.043355e-46 2.426013e-45         1.0000         ****
      Hybrid     Late-EMT  3.779740   1.569923e-04 1.883907e-04         0.6180          ***


In [ ]:
import numpy as np
import pandas as pd

# Your existing 100 raw ID replicates
print(id_reps.shape)
print(id_reps.head())

(100, 4)
   Early-Hybrid  Epithelial     Hybrid   Late-EMT
1     23.194570   22.990030  20.633768  20.577616
2     23.737898   23.455517  21.581519  19.843677
3     22.927989   23.271810  20.524388  20.497941
4     22.989662   23.182688  20.955964  19.910717
5     22.641082   22.728951  21.258430  20.764623


In [ ]:
# ---------------------------------------------------------
# Convert existing raw ID replicates to ID-score
# ---------------------------------------------------------

id_score_reps = (
    id_reps - id_reps.min().min()
) / (
    id_reps.max().max() - id_reps.min().min()
)

print(id_score_reps.head())
print(id_score_reps.shape)

print("\nID-score range:")
print(id_score_reps.min().min(), "to", id_score_reps.max().max())

   Early-Hybrid  Epithelial    Hybrid  Late-EMT
1      0.733444    0.693082  0.228123  0.217043
2      0.840659    0.784936  0.415142  0.072215
3      0.680840    0.748686  0.206539  0.201321
4      0.693010    0.731099  0.291702  0.085444
5      0.624225    0.641564  0.351387  0.253945
(100, 4)

ID-score range:
0.0 to 1.0


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import kruskal, rankdata, norm
from statsmodels.stats.multitest import multipletests
from itertools import combinations


# ============================================================
# SETTINGS
# ============================================================

order = [
    "Epithelial",
    "Early-Hybrid",
    "Hybrid",
    "Late-EMT"
]

data = {
    state: id_score_reps[state].dropna().values
    for state in order
}


# ============================================================
# KRUSKAL-WALLIS
# ============================================================

H, p_kw = kruskal(
    *[data[state] for state in order]
)

print("Kruskal-Wallis")
print("----------------")
print(f"H = {H:.4f}")
print(f"p = {p_kw:.4e}")


# ============================================================
# DUNN TEST
# ============================================================

all_values = np.concatenate(
    [data[state] for state in order]
)

groups = np.concatenate(
    [
        np.repeat(state, len(data[state]))
        for state in order
    ]
)

ranks = rankdata(all_values)

rank_df = pd.DataFrame({
    "group": groups,
    "rank": ranks
})

N = len(all_values)

mean_ranks = (
    rank_df
    .groupby("group")["rank"]
    .mean()
)

n_groups = {
    state: len(data[state])
    for state in order
}


# Tie correction
_, tie_counts = np.unique(
    all_values,
    return_counts=True
)

tie_term = np.sum(
    tie_counts**3 - tie_counts
)

tie_correction = (
    1 -
    tie_term / (N**3 - N)
)


# Pairwise Dunn comparisons
results = []

for g1, g2 in combinations(order, 2):

    n1 = n_groups[g1]
    n2 = n_groups[g2]

    rank_difference = (
        mean_ranks[g1] -
        mean_ranks[g2]
    )

    SE = np.sqrt(
        (
            N * (N + 1) / 12
        )
        *
        (
            1 / n1 +
            1 / n2
        )
        *
        tie_correction
    )

    z = rank_difference / SE

    p = 2 * norm.sf(abs(z))

    results.append({
        "group1": g1,
        "group2": g2,
        "z": z,
        "p_uncorrected": p
    })


dunn_df = pd.DataFrame(results)


# ============================================================
# BENJAMINI-HOCHBERG FDR
# ============================================================

dunn_df["p_FDR"] = multipletests(
    dunn_df["p_uncorrected"],
    method="fdr_bh"
)[1]


# ============================================================
# PRINT STATISTICS
# ============================================================

print("\nDunn pairwise comparisons")
print("=========================")

print(
    dunn_df.to_string(index=False)
)


# ============================================================
# SIGNIFICANCE LABEL
# ============================================================

def significance(p):

    if p < 0.0001:
        return "****"
    elif p < 0.001:
        return "***"
    elif p < 0.01:
        return "**"
    elif p < 0.05:
        return "*"
    else:
        return "ns"


dunn_df["significance"] = (
    dunn_df["p_FDR"].apply(significance)
)


# ============================================================
# BOX PLOT
# ============================================================

fig, ax = plt.subplots(
    figsize=(10, 7)
)

ax.boxplot(
    [
        data[state]
        for state in order
    ],
    patch_artist=True,
    widths=0.6,
    showfliers=False
)


# Individual 100 replicates
rng = np.random.default_rng(42)

for i, state in enumerate(order, start=1):

    values = data[state]

    x = i + rng.uniform(
        -0.05,
        0.05,
        len(values)
    )

    ax.scatter(
        x,
        values,
        s=15,
        alpha=0.35,
        zorder=3
    )


# ============================================================
# SIGNIFICANCE BRACKETS
# ============================================================

ymax = max(
    np.max(data[state])
    for state in order
)

ymin = min(
    np.min(data[state])
    for state in order
)

yrange = ymax - ymin


def add_bracket(
    x1,
    x2,
    y,
    text
):

    h = 0.015 * yrange

    ax.plot(
        [x1, x1, x2, x2],
        [y, y+h, y+h, y],
        color="black",
        linewidth=1.2
    )

    ax.text(
        (x1+x2)/2,
        y+h,
        text,
        ha="center",
        va="bottom",
        fontsize=9
    )


# Put brackets at increasing heights
for i, row in dunn_df.iterrows():

    x1 = order.index(
        row["group1"]
    ) + 1

    x2 = order.index(
        row["group2"]
    ) + 1

    y = (
        ymax
        + (i + 1) * 0.10 * yrange
    )

    add_bracket(
        x1,
        x2,
        y,
        f"{row['significance']}\n"
        f"FDR={row['p_FDR']:.2e}"
    )


# ============================================================
# KRUSKAL-WALLIS ANNOTATION
# ============================================================

ax.text(
    0.5,
    1.02,
    f"Kruskal-Wallis: "
    f"H={H:.2f}, p={p_kw:.2e}",
    transform=ax.transAxes,
    ha="center",
    fontsize=11,
    fontweight="bold"
)


# ============================================================
# LABELS
# ============================================================

ax.set_xticks(
    range(1, 5)
)

ax.set_xticklabels(
    order,
    rotation=30,
    ha="right"
)

ax.set_xlabel("EMT State")
ax.set_ylabel("ID Score")

ax.set_title(
    "ID Score across EMT States",
    fontsize=14
)

ax.set_ylim(
    ymin - 0.05 * yrange,
    ymax + 0.70 * yrange
)

plt.tight_layout()
plt.show()

Kruskal-Wallis
----------------
H = 313.6114
p = 1.1263e-67

Dunn pairwise comparisons
      group1       group2         z  p_uncorrected        p_FDR
  Epithelial Early-Hybrid -0.270331   7.869057e-01 7.869057e-01
  Epithelial       Hybrid 10.207133   1.842287e-24 2.763430e-24
  Epithelial     Late-EMT 13.986873   1.874766e-44 5.624297e-44
Early-Hybrid       Hybrid 10.477464   1.096450e-25 2.192900e-25
Early-Hybrid     Late-EMT 14.257203   4.043355e-46 2.426013e-45
      Hybrid     Late-EMT  3.779740   1.569923e-04 1.883907e-04
